# Where Does Wyoming's Electricity Come From, and Where Does It Go?

An undergraduate lesson using open energy data from the Public Utility Data Liberation project (PUDL), prepared by Noah Parsons. Written as of 13 November 2026.

**Learning goals.** By the end of this notebook you should be able to:

1. load and filter large public energy datasets with pandas;
2. explain the difference between generating *capacity* (MW) and *generation* (MWh), and calculate a capacity factor;
3. describe how Wyoming's generating fleet changed between 2001 and 2025; and
4. estimate how much of Wyoming's electricity is consumed outside the state, and explain the limits of that estimate.

**Data.** PUDL is maintained by Catalyst Cooperative. It cleans and links data that US utilities report to the Energy Information Administration (EIA) and other federal agencies, and publishes it under a CC-BY-4.0 licence. This notebook uses three PUDL tables:

| Table | Source | What it holds |
|---|---|---|
| `out_eia__yearly_generators` | EIA Forms 860 and 923 | One row per generator per year: location, fuel, capacity, status |
| `out_eia923__yearly_generation_fuel_combined` | EIA Form 923 | Net generation per plant, fuel and prime mover per year |
| `core_eia861__yearly_sales` | EIA Form 861 | Retail electricity sales per utility, state and customer class per year |

**Setup.** You need Python 3.10 or later with `pandas`, `pyarrow` and `matplotlib`:

```
pip install pandas pyarrow matplotlib
```

Sections marked **Exercise** are for you to complete. The questions on the worksheet refer to these sections.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:,.1f}".format)

# PUDL's "stable" release. Figures in this lesson were produced from the release
# published on 10 November 2026; a later release may differ slightly.
PUDL = "https://s3.us-west-2.amazonaws.com/pudl.catalyst.coop/stable/{}.parquet"

def load(table, columns=None):
    df = pd.read_parquet(PUDL.format(table), columns=columns)
    for col in df.columns:
        if col.endswith("_date"):
            df[col] = pd.to_datetime(df[col])
    df["year"] = df["report_date"].dt.year
    return df

## 1. Loading the Data

The generator table has more than 600,000 rows and over 100 columns, so we load only the columns we need. This takes a minute or so on a typical connection.

In [ ]:
gens = load("out_eia__yearly_generators", columns=[
    "report_date", "plant_id_eia", "plant_name_eia", "generator_id", "state",
    "utility_name_eia", "balancing_authority_code_eia", "fuel_type_code_pudl",
    "capacity_mw", "operational_status", "operational_status_code", "generator_operating_date",
    "current_planned_generator_operating_date", "planned_generator_retirement_date",
])
gen_fuel = load("out_eia923__yearly_generation_fuel_combined", columns=[
    "report_date", "plant_id_eia", "fuel_type_code_pudl", "net_generation_mwh",
])
sales = load("core_eia861__yearly_sales", columns=[
    "report_date", "utility_id_eia", "state", "customer_class", "service_type", "sales_mwh",
])
print(len(gens), len(gen_fuel), len(sales))

We keep only Wyoming. The generation table has no `state` column, so we identify Wyoming plants from the generator table and filter on plant ID.

In [ ]:
wy_gens = gens[gens["state"] == "WY"]
wy_plant_ids = set(wy_gens["plant_id_eia"])
wy_gen_fuel = gen_fuel[gen_fuel["plant_id_eia"].isin(wy_plant_ids)]
wy_sales = sales[sales["state"] == "WY"]

YEAR = 2025  # the most recent year of final EIA data

## 2. What Wyoming Has: Generating Capacity

*Capacity* is the maximum power a generator can produce, measured in megawatts (MW). We count only generators whose status is `existing`; the table also lists proposed and retired units.

In [ ]:
existing = wy_gens[(wy_gens["year"] == YEAR) & (wy_gens["operational_status"] == "existing")]
capacity = existing.groupby("fuel_type_code_pudl")["capacity_mw"].sum().sort_values(ascending=False)
capacity_table = pd.DataFrame({
    "capacity_mw": capacity,
    "share_%": 100 * capacity / capacity.sum(),
})
capacity_table

**Exercise 2.1.** Which three operators (`utility_name_eia`) own the most existing capacity in Wyoming? Note that one company may report under more than one name.

In [ ]:
# Exercise 2.1: group `existing` by utility_name_eia and sum capacity_mw

## 3. What Wyoming Generates: Generation and Capacity Factor

*Generation* is the energy actually produced over a period, measured in megawatt-hours (MWh). We take it from EIA Form 923, which reports it by plant and fuel.

In [ ]:
generation_twh = (
    wy_gen_fuel[wy_gen_fuel["year"] == YEAR]
    .groupby("fuel_type_code_pudl")["net_generation_mwh"].sum() / 1e6
)
mix = pd.DataFrame({
    "capacity_mw": capacity,
    "generation_twh": generation_twh,
}).fillna(0)
mix["capacity_share_%"] = 100 * mix["capacity_mw"] / mix["capacity_mw"].sum()
mix["generation_share_%"] = 100 * mix["generation_twh"] / mix["generation_twh"].sum()
mix.sort_values("generation_twh", ascending=False)

The **capacity factor** of a fleet is the energy it actually produced divided by the energy it would have produced running at full capacity all year:

$$\text{capacity factor} = \frac{\text{generation (MWh)}}{\text{capacity (MW)} \times 8{,}760\ \text{h}}$$

**Exercise 3.1.** Add a `capacity_factor` column to `mix`. Which fuel has the highest capacity factor, and why might that be?

**Exercise 3.2.** Wyoming's wind capacity grew by almost 1,000 MW during 2025, and some of it came online in September and December. How does that affect the wind capacity factor you calculated? (Hint: look at `generator_operating_date` for wind units.)

In [ ]:
# Exercise 3.1: mix['capacity_factor'] = ...

In [ ]:
# Exercise 3.2: existing[existing['fuel_type_code_pudl'] == 'wind'].sort_values('generator_operating_date').tail(10)

## 4. How the Fleet Has Changed, 2001 to 2025

In [ ]:
capacity_by_year = (
    wy_gens[wy_gens["operational_status"] == "existing"]
    .pivot_table(index="year", columns="fuel_type_code_pudl", values="capacity_mw", aggfunc="sum")
    .fillna(0)
    .loc[2001:YEAR]
)
ax = capacity_by_year[["coal", "wind", "gas", "hydro", "solar"]].plot(
    figsize=(8, 4.5), style=["-", "--", "-.", ":", "-"], color="black")
ax.set_ylabel("Existing capacity (MW)")
ax.set_title("Wyoming generating capacity by fuel")
plt.show()
capacity_by_year.loc[[2001, 2010, 2020, 2023, 2024, YEAR], ["coal", "gas", "wind", "solar", "hydro"]]

**Exercise 4.1.** Coal capacity falls and gas capacity rises sharply between 2023 and 2024. Find the generators responsible by comparing each generator's `fuel_type_code_pudl` in 2023 and 2024. What happened to them?

**Exercise 4.2.** A smaller version of the same change happens between 2019 and 2020. Which plant is involved?

In [ ]:
# Exercise 4.1: one approach
# both = wy_gens[wy_gens["year"].isin([2023, 2024]) & (wy_gens["operational_status"] == "existing")]
# fuel = both.pivot_table(index=["plant_name_eia", "generator_id"], columns="year",
#                         values="fuel_type_code_pudl", aggfunc="first")
# fuel[fuel[2023] != fuel[2024]]

## 5. Where It Goes: Generation Compared with In-State Sales

EIA Form 861 records retail sales to customers in each state. Comparing Wyoming's total generation with its retail sales gives a rough estimate of how much electricity leaves the state.

Form 861 reports some sales as `bundled` (energy and delivery from one utility), `energy` only, or `delivery` only. Adding `delivery` rows to the others would count the same energy twice, so we leave them out.

In [ ]:
gen_by_year = wy_gen_fuel.groupby("year")["net_generation_mwh"].sum() / 1e6
sales_by_year = (
    wy_sales[wy_sales["service_type"].isin(["bundled", "energy"])]
    .groupby("year")["sales_mwh"].sum() / 1e6
)
balance = pd.DataFrame({"generation_twh": gen_by_year, "retail_sales_twh": sales_by_year}).loc[2001:YEAR]
balance["net_export_share_%"] = 100 * (1 - balance["retail_sales_twh"] / balance["generation_twh"])

ax = balance[["generation_twh", "retail_sales_twh"]].plot(figsize=(8, 4.5), style=["-", "--"], color="black")
ax.set_ylabel("TWh per year")
ax.set_title("Wyoming generation and in-state retail sales")
plt.show()
balance.loc[[2001, 2010, 2020, 2024, YEAR]]

**Exercise 5.1.** Roughly what share of the electricity generated in Wyoming in 2025 was not sold to Wyoming customers?

**Exercise 5.2.** This is a *net* figure and an approximation. Name at least two reasons why the true share of Wyoming's generation consumed elsewhere could differ from it.

## 6. What Is Coming: Proposed Capacity and Retirements

PUDL's `proposed` status includes every generator EIA lists on its proposed schedule, including projects that the
developer has indefinitely postponed (EIA status code `IP`) or cancelled (`CN`). We keep only the active codes:
`P` (planned, approvals not initiated), `L` (approvals pending), `T` (approvals received), `U` and `V` (under
construction), `TS` (built but not yet in commercial operation) and `OT` (other).

In [ ]:
ACTIVE = ["P", "L", "T", "U", "V", "TS", "OT"]
all_proposed = wy_gens[(wy_gens["year"] == YEAR) & (wy_gens["operational_status"] == "proposed")]
proposed = all_proposed[all_proposed["operational_status_code"].isin(ACTIVE)]
print("Postponed or cancelled:", (~all_proposed["operational_status_code"].isin(ACTIVE)).sum(), "generators,",
      round(all_proposed.loc[~all_proposed["operational_status_code"].isin(ACTIVE), "capacity_mw"].sum(), 1), "MW")
(proposed.groupby("fuel_type_code_pudl")
    .agg(generators=("generator_id", "count"),
         capacity_mw=("capacity_mw", "sum"),
         earliest=("current_planned_generator_operating_date", "min"),
         latest=("current_planned_generator_operating_date", "max"))
    .sort_values("capacity_mw", ascending=False))

In [ ]:
existing[existing["planned_generator_retirement_date"].notna()][
    ["plant_name_eia", "generator_id", "fuel_type_code_pudl", "capacity_mw",
     "planned_generator_retirement_date", "utility_name_eia"]]

**Exercise 6.1.** If every active proposed project were built, by how much would Wyoming's existing capacity grow? Why is it unlikely that all of it will be built?

**Exercise 6.2.** Most proposed capacity is wind and solar. Using what you found in Sections 3 and 5, explain why new transmission matters for these projects.

## 7. Limits of the Data

Before drawing conclusions, consider:

- **Nameplate capacity** is a design rating; the capacity available in summer or winter is usually lower.
- **Fuel codes describe the primary fuel.** Some plants burn more than one fuel, and conversions (Section 4) change a unit's fuel from one year to the next.
- **Proposed projects** are reported by developers to EIA. Many are delayed or cancelled.
- **Retail sales** exclude energy lost in transmission and distribution, and do not show which plants served which customers.

## Acknowledgement

Data from the Public Utility Data Liberation project, Catalyst Cooperative, licensed under CC-BY-4.0. See [catalyst.coop/pudl](https://catalyst.coop/pudl/) for documentation and citation guidance.